# 🧠 Module 2 – Session 1 Assignment
# Investigating Attention in BERT

**Course:** Generative & Agentic AI Systems

## Objective
In this assignment you will investigate how BERT distributes attention across different linguistic phenomena.

Rather than proving *how BERT thinks*, your goal is to collect evidence and make engineering observations.

---
## Learning Outcomes
After completing this notebook you should be able to:

- Build linguistic probes
- Visualize attention using BertViz
- Identify interesting attention heads
- Compare expectations with observations
- Write an engineering conclusion


# Part 0 – Environment

In [1]:
# Uncomment if needed
!pip install transformers bertviz torch

  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached bertviz-1.4.1-py3-none-any.whl.metadata (19 kB)
  Using cached torch-2.13.0-cp313-cp313-win_amd64.whl.metadata (39 kB)
  Using cached huggingface_hub-1.26.0-py3-none-any.whl.metadata (16 kB)
  Using cached numpy-2.5.1-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.7.19-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.27.0-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.32.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.2-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-


[notice] A new release of pip is available: 25.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME="bert-base-uncased"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModel.from_pretrained(MODEL_NAME,output_attentions=True)
model.eval()

print("Model Loaded Successfully")

e:\DEPI round5 tasks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\DEPI round5 tasks\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\esraa\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this ar

Model Loaded Successfully


In [3]:
def get_attention(sentence):
    inputs=tokenizer(sentence,return_tensors="pt")
    with torch.no_grad():
        outputs=model(**inputs)

    tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return tokens,outputs.attentions

# Part 1 – Build Your Test Set (15 Marks)

Create **5 sentences**.

| Sentence Type | Your Sentence | Expected Attention |
|---------------|---------------|--------------------|
| Long Dependency | The book that my professor recommended yesterday was fascinating.| book->was->fascinating->professor->recomended|
| Ambiguous Pronoun |sarah told emma that she won the prize |she, sara, emma |
| Negation | i do not like pizza |not->like->pizza |
| Passive Voice |the report was writen by manager |report->writen->manager|
| Free Choice | Artificial intelligence is transforming healthcare. |Artificial intelligence, transforming healthcare |

**Questions**

1. Why did you choose these sentences? because every sentence represent different linguitic challenge for BERT
2. What do you expect BERT to focus on? i excpect to focus on words that carry the meanning in the sentence like verb and subject or negation words and important pronouns


# Part 2 – Run Your Experiments

Sentence One

In [4]:
# Replace with one of your own sentences

sentence="The book that my professor recommended yesterday was fascinating."

tokens,attentions=get_attention(sentence)

print(tokens)
print("Layers:",len(attentions))
print("Heads:",attentions[0].shape[1])

['[CLS]', 'the', 'book', 'that', 'my', 'professor', 'recommended', 'yesterday', 'was', 'fascinating', '.', '[SEP]']
Layers: 12
Heads: 12


In [5]:
from bertviz import head_view
head_view(attentions,tokens)

# Take a screenshot and paste it below.

<IPython.core.display.Javascript object>

In [6]:
# TODO

from bertviz import model_view
model_view(attentions,tokens)

# Take another screenshot.


<IPython.core.display.Javascript object>

# Part 3 – Head Investigation (25 Marks)

Complete this table.

| Layer | Head | Behavior | Evidence | Did it match your expectation? |
|-------|------|----------|----------|--------------------------------|
|0 | 5| positional |professor most attended to my because it is the previous token|no |
| 0| 11| Long-range | the word book more attended to fascinating not was | no|
| 9| 3 | Special Token / CLS |"book"mave most attention to SEP|no|

After the table, answer:

- Why is this head interesting? it focus on the ability of BERT to positional nearby words, also for layer 0 and head 11 it shows that BERT making connections between words that are not adjacent, which is vital for understanding the overall meaning
- Which tokens communicate?1- my with professor, 2-book with fascinating, 3-book is communicating with the SEP
- What engineering insight does it provide?Early layers in BERT seem to have heads dedicated to capturing immediate, local grammatical relationships. This suggests that basic syntactic parsing happens early in the model's processing pipeline.


Sentence Two

In [7]:
# Replace with one of your own sentences

sentence="sarah told emma that she won the prize"

tokens,attentions=get_attention(sentence)

print(tokens)
print("Layers:",len(attentions))
print("Heads:",attentions[0].shape[1])

['[CLS]', 'sarah', 'told', 'emma', 'that', 'she', 'won', 'the', 'prize', '[SEP]']
Layers: 12
Heads: 12


In [8]:
from bertviz import head_view
head_view(attentions,tokens)

# Take a screenshot and paste it below.

<IPython.core.display.Javascript object>

In [9]:
# TODO

from bertviz import model_view
model_view(attentions,tokens)

# Take another screenshot.


<IPython.core.display.Javascript object>

# Part 3 – Head Investigation (25 Marks)

Complete this table.

| Layer | Head | Behavior | Evidence | Did it match your expectation? |
|-------|------|----------|----------|--------------------------------|
|3| 3| positional |emma give the most attention to told(the previous word)|no |
| 3| 5| Long-range | Sarah take the most attention from she | yes|
| 3| 0:11 | Special Token / CLS |all tokens give the most attention to SEP and CLS|no|

After the table, answer:

- Why is this head interesting?
    *   **Layer 3, Head 3 (Positional):** This head is interesting because it effectively captures local, direct grammatical relationships, specifically `emma` attending strongly to the preceding verb `told`. This highlights BERT's ability to identify immediate syntactic dependencies.
    *   **Layer 3, Head 5 (Long-range):** This head is interesting as it demonstrates BERT's capacity to resolve anaphoric references by connecting `she` back to `Sarah` across intervening tokens. This is a crucial mechanism for understanding coherent text.
    *   **Layer 3, 0:11 (Special Token / CLS):** These heads are interesting because they show a broad aggregation of information from all tokens towards the `[SEP]` and `[CLS]` tokens. This indicates their role in summarizing sentence-level context or marking sentence boundaries, fundamental for overall input understanding.

- Which tokens communicate?
    *   **Layer 3, Head 3:** `emma` communicates with `told`.
    *   **Layer 3, Head 5:** `she` communicates with `Sarah`.
    *   **Layer 3, 0:11:** All other tokens (`[CLS]`, `sarah`, `told`, `emma`, `that`, `she`, `won`, `the`, `prize`) communicate with the `[SEP]` and `[CLS]` tokens.

- What engineering insight does it provide?
    *   **Layer 3, Head 3 (Positional):** This provides insight that even in intermediate layers, BERT maintains heads focused on immediate, local grammatical relationships, suggesting a continuous refinement of syntactic understanding throughout the model's depth.
    *   **Layer 3, Head 5 (Long-range):** This demonstrates that BERT has specialized heads for resolving long-distance dependencies, particularly pronoun resolution. This is a powerful feature for understanding complex sentence structures and coreference.
    *   **Layer 3, 0:11 (Special Token / CLS):** The insight here is that BERT dedicates certain heads to creating sentence-level representations by strongly attending to special tokens like `[CLS]` and `[SEP]`. This aggregation is vital for tasks that rely on a summary of the entire input, such as classification or entailment.

Sentence Three

In [10]:
# Replace with one of your own sentences

sentence="i do not like pizza"

tokens,attentions=get_attention(sentence)

print(tokens)
print("Layers:",len(attentions))
print("Heads:",attentions[0].shape[1])

['[CLS]', 'i', 'do', 'not', 'like', 'pizza', '[SEP]']
Layers: 12
Heads: 12


In [11]:
from bertviz import head_view
head_view(attentions,tokens)

# Take a screenshot and paste it below.

<IPython.core.display.Javascript object>

In [12]:
# TODO

from bertviz import model_view
model_view(attentions,tokens)

# Take another screenshot.


<IPython.core.display.Javascript object>

# Part 3 – Head Investigation (25 Marks)

Complete this table.

| Layer | Head | Behavior | Evidence | Did it match your expectation? |
|-------|------|----------|----------|--------------------------------|
|11 | 9| positional |not gets more attention from tokens that near for it |no |
| 11| 9| Long-range | pizza gets attention from i and CLS | no|
| 11| 9 | Special Token / CLS |every token give the most attention for SEP|no|

After the table, answer:

- Why is this head interesting?
    *   **Layer 11, Head 9 (Positional - Negation):** This head is interesting because it shows a clear focus on the negation word 'not' and its immediate neighbors ('do', 'like'). This indicates a specialized mechanism for understanding how negation modifies the verb's meaning, which is crucial for correctly interpreting the sentence's sentiment or intent.
    *   **Layer 11, Head 9 (Long-range - Subject-Object):** This aspect of the head is interesting as it connects the object 'pizza' with the subject 'i' and the overall sentence representation `[CLS]`. This demonstrates the model's ability to establish core semantic relationships across intervening words, which is vital for understanding who is doing what to whom.
    *   **Layer 11, Head 9 (Special Token / CLS - Sentence Boundary):** The observation that every token gives most attention to `[SEP]` for this head is interesting because it suggests a role in signaling the end of the sentence or aggregating all token information towards the final sentence boundary token. This is important for tasks that require a complete sentence-level understanding.

- Which tokens communicate?
    *   **For Positional behavior:** `not` communicates with `do` and `like`.
    *   **For Long-range behavior:** `pizza` communicates with `i` and `[CLS]`.
    *   **For Special Token / CLS behavior:** All tokens (`[CLS]`, `i`, `do`, `not`, `like`, `pizza`) communicate with `[SEP]`.

- What engineering insight does it provide?
    *   **For Positional behavior:** This provides insight that even in the deepest layers, specific heads are dedicated to processing critical local modifiers like negation. This fine-grained attention to function words helps BERT accurately capture semantic nuances.
    *   **For Long-range behavior:** This demonstrates that later layers develop heads that integrate information from distant but semantically related tokens (subject-object) and central summary tokens (`[CLS]`). This is key for robust sentence understanding.
    *   **For Special Token / CLS behavior:** The insight here is that some heads in the final layers contribute to consolidating sentence-level information by strongly attending to `[SEP]`. This aggregation is vital for generating a final context-rich representation of the entire input.

Sentence Four

In [13]:
# Replace with one of your own sentences

sentence="the report was writen by manager"

tokens,attentions=get_attention(sentence)

print(tokens)
print("Layers:",len(attentions))
print("Heads:",attentions[0].shape[1])

['[CLS]', 'the', 'report', 'was', 'write', '##n', 'by', 'manager', '[SEP]']
Layers: 12
Heads: 12


In [14]:
from bertviz import head_view
head_view(attentions,tokens)

# Take a screenshot and paste it below.

<IPython.core.display.Javascript object>

In [15]:
# TODO

from bertviz import model_view
model_view(attentions,tokens)

# Take another screenshot.


<IPython.core.display.Javascript object>

# Part 3 – Head Investigation (25 Marks)

Complete this table.

| Layer | Head | Behavior | Evidence | Did it match your expectation? |
|-------|------|----------|----------|--------------------------------|
|5 | 0:11| positional |every token gets more attention from the previous token|no |
| 5| 0:11| Long-range | manager gets attention from all tokens but the most was from by | no|
| 9| 3 | Special Token / CLS |every token gives attention the most for SEP|no|

After the table, answer:

- Why is this head interesting?
    *   **Layer 5, Head 0:11 (Positional):** This head is interesting because it consistently shows local, sequential attention, where tokens primarily attend to their immediate preceding token. This highlights a foundational mechanism for building an understanding of word order and basic grammatical structures within the sentence.
    *   **Layer 5, Head 0:11 (Long-range - Passive Agent):** This head is interesting as it demonstrates BERT's ability to identify and connect the agent (`manager`) in a passive voice construction to the preposition (`by`) that introduces it. This is crucial for correctly identifying who performs the action, even when the sentence structure is inverted.
    *   **Layer 9, Head 3 (Special Token / CLS - Sentence Boundary):** This head is interesting because its strong and broad attention to the `[SEP]` token from nearly all other tokens indicates a role in signaling the end of the input sequence or aggregating sentence-level information towards the boundary token. This is vital for tasks relying on complete sentence context.

- Which tokens communicate?
    *   **For Positional behavior (Layer 5, Head 0:11):** Each token primarily communicates with its immediately preceding token (e.g., `report` with `the`, `was` with `report`, `manager` with `by`).
    *   **For Long-range behavior (Layer 5, Head 0:11):** `manager` primarily communicates with `by`, and to a lesser extent, with other tokens in the sentence.
    *   **For Special Token / CLS behavior (Layer 9, Head 3):** All tokens (`[CLS]`, `the`, `report`, `was`, `write`, `##n`, `by`, `manager`) communicate strongly with `[SEP]`.

- What engineering insight does it provide?
    *   **For Positional behavior (Layer 5, Head 0:11):** This suggests that mid-level layers in BERT maintain heads focused on basic sequential processing and local grammatical links. This builds a robust initial understanding of sentence structure before more complex semantic relationships are formed.
    *   **For Long-range behavior (Layer 5, Head 0:11):** This provides insight into BERT's capacity to handle passive voice constructions by effectively linking the agent to its introductory preposition. This capability is essential for accurate semantic role labeling and event understanding, which are critical for many NLP applications.
    *   **For Special Token / CLS behavior (Layer 9, Head 3):** The insight here is that heads in deeper layers contribute to creating a definitive end-of-sequence signal by attending heavily to `[SEP]`. This mechanism is important for downstream tasks that rely on clear sentence demarcation or require a holistic summary of the input.

Sentence Five

In [16]:
# Replace with one of your own sentences

sentence="Artificial intelligence is transforming healthcare."

tokens,attentions=get_attention(sentence)

print(tokens)
print("Layers:",len(attentions))
print("Heads:",attentions[0].shape[1])

['[CLS]', 'artificial', 'intelligence', 'is', 'transforming', 'healthcare', '.', '[SEP]']
Layers: 12
Heads: 12


In [17]:
from bertviz import head_view
head_view(attentions,tokens)

# Take a screenshot and paste it below.

<IPython.core.display.Javascript object>

In [18]:
# TODO

from bertviz import model_view
model_view(attentions,tokens)

# Take another screenshot.


<IPython.core.display.Javascript object>

# Part 3 – Head Investigation (25 Marks)

Complete this table.

| Layer | Head | Behavior | Evidence | Did it match your expectation? |
|-------|------|----------|----------|--------------------------------|
|10 | 3| positional |Intelligence recieves little attention from Artificial|no |
| 10| 0| Long-range | is gets the most attention from CLS that for from it | no|
| 10| 1 | Special Token / CLS |every token give most attention to SEP|no|

After the table, answer:

- Why is this head interesting?
    *   **Layer 10, Head 3 (Positional):** This head is interesting because it *defies* the expectation of strong local attention between adjacent words forming a compound noun ('Artificial intelligence'). The evidence 'Intelligence receives little attention from Artificial' suggests this head might prioritize other types of relationships or that its 'positional' focus is more complex than simple adjacency for all phrases.
    *   **Layer 10, Head 0 (Long-range):** This head is interesting as it demonstrates a long-range dependency where the auxiliary verb `is` receives significant attention from the `[CLS]` token. This suggests a role in aggregating information about the sentence's predicate or main verb into the overall sentence representation.
    *   **Layer 10, Head 1 (Special Token / CLS):** This head is interesting because it shows all tokens giving most attention to the `[SEP]` token. This indicates a strong focus on sentence boundary detection and the consolidation of information towards the end-of-sequence marker, which is crucial for defining the scope of the input.

- Which tokens communicate?
    *   **For Positional behavior (Layer 10, Head 3):** Based on the evidence, `Intelligence` does not strongly communicate with `Artificial` in a direct positional sense within this head.
    *   **For Long-range behavior (Layer 10, Head 0):** `is` communicates strongly with `[CLS]`.
    *   **For Special Token / CLS behavior (Layer 10, Head 1):** All tokens (`[CLS]`, `artificial`, `intelligence`, `is`, `transforming`, `healthcare`, `.`) communicate strongly with `[SEP]`.

- What engineering insight does it provide?
    *   **For Positional behavior (Layer 10, Head 3):** This suggests that not all heads labeled 'positional' will uniformly capture simple adjacency for all word combinations. BERT might distribute positional encoding responsibilities across various heads, some focusing on different aspects of local structure or interacting with other features.
    *   **For Long-range behavior (Layer 10, Head 0):** This provides insight that even in deeper layers, specific heads are designed to connect key functional words (like auxiliary verbs) to the `[CLS]` token. This aggregation of predicate information is vital for forming a holistic sentence embedding.
    *   **For Special Token / CLS behavior (Layer 10, Head 1):** This demonstrates that heads in later layers are actively involved in establishing and reinforcing sentence boundaries. This mechanism is crucial for tasks that require clear demarcation of input sequences or when processing multiple sentences.

# Part 4 – Expectations vs Reality (15 Marks)

| Sentence | Expected | Observed | Match? |
|-----------|----------|----------|--------|
| 1. Long Dependency | `book`->`was`->`fascinating`->`professor`->`recommended` | Positional: `professor`->`my`; Long-range: `book`->`fascinating`; Special Token: `book`->`SEP` | No |
| 2. Ambiguous Pronoun | `she` resolves to `sarah` or `emma` | Long-range: `she`->`Sarah` | Yes |
| 3. Negation | `not`->`like`->`pizza` | Positional: `not`->`do`, `like`; Long-range: `pizza`->`i`, `[CLS]` | Partial |
| 4. Passive Voice | `report`->`written`->`manager` | Long-range: `manager`->`by` | Partial |
| 5. Free Choice | `Artificial intelligence`, `transforming healthcare` | Positional: `Intelligence` has *little* attention from `Artificial` | No |

Write a short discussion (100–150 words) explaining any surprising findings.

**Discussion:**

My initial expectations for how BERT would distribute attention were sometimes met, but often revealed more nuanced and complex patterns. For instance, the pronoun resolution in Sentence Two (`she` -> `Sarah`) largely matched, demonstrating BERT's ability to handle coreference. Similarly, the negation in Sentence Three showed `not` influencing `like`, aligning with the core expectation.

However, the results for Sentence One and Five were particularly surprising. For the long dependency in Sentence One, I expected direct connections between key semantic elements, but instead observed more fragmented attention, with `book` attending to `fascinating` rather than `was` or `recommended`. Even more unexpectedly, in Sentence Five, the compound noun `Artificial intelligence` did not show strong immediate positional attention between its components (`Artificial` and `intelligence`), suggesting that BERT might parse such phrases using distributed attention across multiple heads or layers rather than a single, obvious connection. These findings highlight that attention mechanisms are not always straightforward interpretations of human linguistic intuition.

# Part 5 – Engineering Reflection (20 Marks)

Imagine your manager asks:

> **Should engineers use attention maps for debugging production models?**

Write **300–400 words** addressing:

1. What attention maps reveal well.
2. Their limitations.
3. Why attention is **not** a complete explanation.
4. When engineers should use them.
5. Your final recommendation with a confidence level.

**Response:**

**1. What Attention Maps Reveal Well:** Attention maps are powerful visualization tools that offer a window into how transformer-based models like BERT process input sequences. They excel at revealing token-level dependencies, highlighting which parts of the input a model focuses on when processing another. This can illuminate syntactic structures (e.g., subject-verb agreement), semantic relationships (e.g., coreference resolution), and even the influence of special tokens (`[CLS]`, `[SEP]`). They provide intuitive, human-readable insights into patterns of information flow and can pinpoint specific heads responsible for certain linguistic phenomena.

**2. Their Limitations:** Despite their utility, attention maps have significant limitations. Firstly, they often present a highly granular view, making it challenging to extract higher-level, holistic insights without extensive manual analysis. The sheer number of heads and layers can lead to an overwhelming amount of data. Secondly, attention patterns can be complex and sometimes defy straightforward linguistic interpretation, as seen in our experiments where expected connections were not always direct or singular. Moreover, attention weights are not always perfectly correlated with feature importance or causal influence on the model's output.

**3. Why Attention is Not a Complete Explanation:** Attention is a mechanism for information aggregation and transformation within the transformer architecture, but it is not a complete explanation for a model's decision-making process. The final prediction relies on a complex interplay of attention, feed-forward networks, residual connections, and layer normalization across multiple layers. A high attention weight doesn't necessarily mean a token is causally important for the output; it merely indicates that information from that token was heavily considered. Other factors, like the values within the query, key, and value matrices, and subsequent non-linear transformations, also play crucial roles.

**4. When Engineers Should Use Them:** Engineers should primarily use attention maps for **qualitative debugging and hypothesis generation**. They are invaluable for:
    *   **Initial model understanding:** Gaining an intuition for how the model handles specific linguistic structures.
    *   **Identifying unexpected behavior:** Pinpointing cases where the model focuses on irrelevant tokens or fails to capture critical dependencies.
    *   **Guiding feature engineering:** If a model struggles with a certain type of input, attention maps can suggest which features might be missing or underutilized.
    *   **Educational purposes:** Explaining transformer mechanics to non-experts.

**5. Final Recommendation with a Confidence Level:**

My recommendation is to **use attention maps as a diagnostic aid, not a definitive causal explanation**, with a **confidence level of 7/10**. While they offer valuable qualitative insights into model internals and can help identify potential issues, engineers should **not** rely on them as the sole or primary debugging tool for production models. They are best employed as part of a broader interpretability toolkit, alongside techniques like integrated gradients, LIME, or SHAP, which provide more direct insights into feature importance and causal contributions to predictions. For robust production debugging, a combination of these methods, coupled with rigorous quantitative evaluation and error analysis, is essential.

# ⭐ Bonus (+10)

Try one of the following:

- Winograd sentence
- Sarcasm
- Idiom
- Arabic sentence
- Code-mixed sentence
- Emoji sentence

Discuss whether attention changed.


# ✅ Submission Checklist

- [ ] Five original sentences
- [ ] Predictions written before experiments
- [ ] BertViz screenshots included
- [ ] Three head behaviors documented
- [ ] Expectations vs Reality completed
- [ ] Engineering reflection completed
- [ ] Notebook executed from start to finish
